# Analisis Exploratorio de Datos (EDA): Prediccion de Retrasos Logisticos

Proyecto Final - MLOps

## Objetivo
Analizar la estructura del dataset logistico, evaluar su calidad y extraer hallazgos para modelar la variable objetivo `Late_delivery_risk` en una siguiente fase de baseline.

## 1. Importacion de librerias

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 2. Carga y vista general de los datos

In [2]:
primary_path = Path("../data/raw/DataCoSupplyChainDataset.csv")
fallback_path = Path("../data/raw/supply_chain_data.csv")

if primary_path.exists():
    DATA_PATH = primary_path
elif fallback_path.exists():
    DATA_PATH = fallback_path
else:
    raise FileNotFoundError("No se encontro DataCoSupplyChainDataset.csv ni supply_chain_data.csv en ../data/raw")

df = pd.read_csv(DATA_PATH, encoding="latin-1", low_memory=False)
df.columns = [c.strip() for c in df.columns]

print(f"Archivo: {DATA_PATH}")
print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head(10)

Archivo: ..\data\raw\DataCoSupplyChainDataset.csv
Dataset cargado: 180519 filas x 53 columnas


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class
5,TRANSFER,6,4,18.580000,294.980011,Shipping canceled,0,73,Sporting Goods,Tonawanda,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/19/2018 11:03,Standard Class
6,DEBIT,2,1,95.180000,288.420013,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 10:42,First Class
7,TRANSFER,2,1,68.430000,285.140015,Late delivery,1,73,Sporting Goods,Miami,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 10:21,First Class
8,CASH,3,2,133.720001,278.589996,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 10:00,Second Class
9,CASH,2,1,132.149994,275.309998,Late delivery,1,73,Sporting Goods,San Ramon,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 9:39,First Class


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer Fname                

In [4]:
print("Resumen estadistico (numericas):")
display(df.describe().T)

print("\nResumen de variables categoricas:")
display(df.describe(include="object").T)

print("\nValores unicos por columna (top 20):")
display(df.nunique().sort_values(ascending=False).head(20))

Resumen estadistico (numericas):


,count,mean,std,min,25%,50%,75%,max
Days for shipping (real),180519.0,3.497654,1.623722,0.000000,2.000000,3.000000,5.000000,6.000000
Days for shipment (scheduled),180519.0,2.931847,1.374449,0.000000,2.000000,4.000000,4.000000,4.000000
Benefit per order,180519.0,21.974989,104.433526,-4274.979980,7.000000,31.520000,64.800003,911.799988
Sales per customer,180519.0,183.107609,120.043670,7.490000,104.379997,163.990005,247.399994,1939.989990
Late_delivery_risk,180519.0,0.548291,0.497664,0.000000,0.000000,1.000000,1.000000,1.000000
Category Id,180519.0,31.851451,15.640064,2.000000,18.000000,29.000000,45.000000,76.000000
Customer Id,180519.0,6691.379495,4162.918106,1.000000,3258.500000,6457.000000,9779.000000,20757.000000
Customer Zipcode,180516.0,35921.126914,37542.461122,603.000000,725.000000,19380.000000,78207.000000,99205.000000
Department Id,180519.0,5.443460,1.629246,2.000000,4.000000,5.000000,7.000000,12.000000
Latitude,180519.0,29.719955,9.813646,-33.937553,18.265432,33.144863,39.279617,48.781933



Resumen de variables categoricas:


,count,unique,top,freq
Type,180519,4,DEBIT,69295
Delivery Status,180519,4,Late delivery,98977
Category Name,180519,50,Cleats,24551
Customer City,180519,563,Caguas,66770
Customer Country,180519,2,EE. UU.,111146
Customer Email,180519,1,XXXXXXXXX,180519
Customer Fname,180519,782,Mary,65150
Customer Lname,180511,1109,Smith,64104
Customer Password,180519,1,XXXXXXXXX,180519
Customer Segment,180519,3,Consumer,93504



Valores unicos por columna (top 20):


Order Item Id                 180519
order date (DateOrders)        65752
Order Id                       65752
shipping date (DateOrders)     63701
Benefit per order              21998
Order Profit Per Order         21998
Customer Id                    20652
Order Customer Id              20652
Latitude                       11250
Customer Street                 7458
Longitude                       4487
Order City                      3597
Order Item Total                2927
Sales per customer              2927
Customer Lname                  1109
Order State                     1089
Order Item Discount             1017
Customer Zipcode                 995
Customer Fname                   782
Order Zipcode                    609
dtype: int64

## 3. Calidad de datos y consistencia

In [5]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)

quality = pd.DataFrame({"nulos": null_counts, "pct_nulos": null_pct})
quality = quality.sort_values("pct_nulos", ascending=False)

print("Columnas con nulos:")
display(quality[quality["nulos"] > 0])

print(f"Filas duplicadas exactas: {df.duplicated().sum()}")

numeric_cols = df.select_dtypes(include=np.number).columns
negative_counts = (df[numeric_cols] < 0).sum().sort_values(ascending=False)
print("\nColumnas numericas con valores negativos (top 15):")
display(negative_counts[negative_counts > 0].head(15))

Columnas con nulos:


,nulos,pct_nulos
Product Description,180519,100.00
Order Zipcode,155679,86.24
Customer Lname,8,0.00
Customer Zipcode,3,0.00


Filas duplicadas exactas: 0

Columnas numericas con valores negativos (top 15):


Longitude                  180414
Benefit per order           33784
Order Item Profit Ratio     33784
Order Profit Per Order      33784
Latitude                        9
dtype: int64

Notas de calidad de datos:
- El dataset no presenta duplicados exactos a nivel de fila.
- Existen columnas con nulos altos que deben tratarse segun su relevancia para modelado.


## 4. Analisis de la variable objetivo

In [6]:
target_col = "Late_delivery_risk"

if target_col not in df.columns:
    raise ValueError("No se encontro la columna Late_delivery_risk en el dataset")

target_counts = df[target_col].value_counts().sort_index()
target_pct = (df[target_col].value_counts(normalize=True).sort_index() * 100).round(2)

labels = {0: "No retraso (0)", 1: "Riesgo de retraso (1)"}
x_labels = [labels.get(x, str(x)) for x in target_counts.index]

fig = px.bar(
    x=x_labels,
    y=target_counts.values,
    text=target_counts.values,
    color=x_labels,
    labels={"x": "Clase", "y": "Cantidad"},
    title="Distribucion de Late_delivery_risk"
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

print("Distribucion del target:")
for k in target_counts.index:
    print(f"{labels.get(k, k)}: {target_counts[k]:,} registros ({target_pct[k]:.2f}%)")

Distribucion del target:
No retraso (0): 81,542 registros (45.17%)
Riesgo de retraso (1): 98,977 registros (54.83%)


## 5. Variables temporales, correlaciones y factores explicativos

In [7]:
df_corr = df.copy()

if "order date (DateOrders)" in df_corr.columns:
    df_corr["order_date"] = pd.to_datetime(df_corr["order date (DateOrders)"], errors="coerce")
    df_corr["order_month"] = df_corr["order_date"].dt.month
    df_corr["order_dow"] = df_corr["order_date"].dt.dayofweek

if "shipping date (DateOrders)" in df_corr.columns and "order_date" in df_corr.columns:
    df_corr["shipping_date"] = pd.to_datetime(df_corr["shipping date (DateOrders)"], errors="coerce")
    df_corr["delivery_days"] = (df_corr["shipping_date"] - df_corr["order_date"]).dt.total_seconds() / 86400

numeric_cols = df_corr.select_dtypes(include=np.number).columns.tolist()

if target_col in numeric_cols:
    corr_target = df_corr[numeric_cols].corr()[target_col].drop(target_col).sort_values(key=abs, ascending=False)
    top_corr_cols = corr_target.head(12).index.tolist() + [target_col]
    corr_matrix = df_corr[top_corr_cols].corr()

    fig = px.imshow(
        corr_matrix,
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
        text_auto=".2f",
        title="Matriz de correlacion (top variables numericas respecto al target)"
    )
    fig.update_layout(height=700)
    fig.show()
else:
    corr_target = pd.Series(dtype=float)
    print("No fue posible calcular correlaciones con el target.")

In [8]:
if len(corr_target) > 0:
    print("Top correlaciones absolutas con Late_delivery_risk:")
    display(corr_target.head(15).to_frame("corr").style.format({"corr": "{:.3f}"}))

if "Shipping Mode" in df.columns:
    delay_rate_by_ship = df.groupby("Shipping Mode")[target_col].mean().sort_values(ascending=False)
    print("\nTasa de retraso por modo de envio:")
    display((delay_rate_by_ship * 100).round(2).to_frame("pct_retraso"))

Top correlaciones absolutas con Late_delivery_risk:


,corr
Days for shipping (real),0.401
delivery_days,0.387
Days for shipment (scheduled),-0.369
Order Zipcode,-0.014
Sales per customer,-0.004
Order Item Total,-0.004
Order Profit Per Order,-0.004
Benefit per order,-0.004
Sales,-0.004
Customer Zipcode,0.003



Tasa de retraso por modo de envio:


,pct_retraso
Shipping Mode,
First Class,95.32
Second Class,76.63
Same Day,45.74
Standard Class,38.07


In [9]:
country_counts = df["Order Country"].value_counts().reset_index()
country_counts.columns = ["Pais", "Pedidos"]

fig_map = px.choropleth(
    country_counts,
    locations="Pais",
    locationmode="country names",
    color="Pedidos",
    color_continuous_scale="YlOrRd",
    title="Distribucion de pedidos por pais"
 )
fig_map.update_layout(geo=dict(showframe=False, showcoastlines=True), height=500)
fig_map.show()

print(f"Total de paises con pedidos: {country_counts['Pais'].nunique()}")
print("Top 10 paises por volumen:")
display(country_counts.head(10))

Total de paises con pedidos: 164
Top 10 paises por volumen:


,Pais,Pedidos
0,Estados Unidos,24840
1,Francia,13222
2,México,13172
3,Alemania,9564
4,Australia,8497
5,Brasil,7987
6,Reino Unido,7302
7,China,5758
8,Italia,4989
9,India,4783


Conclusiones del EDA

1. El dataset presenta volumen suficiente para modelado supervisado de retrasos.
2. La variable objetivo tiene estructura binaria y permite evaluar metricas de clasificacion.
3. Se identifican variables temporales, operativas y geograficas con potencial explicativo.
